In [1]:
import arff
import pandas as pd

# загружаем данные
with open("c:/Personal/Science/Data/Binary_classification/road-safety.arff", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],
                  columns=[a[0] for a in dataset["attributes"]])
pd.set_option("display.max_columns", None)
df.head()

,Vehicle_Reference_df_res,Vehicle_Type,Vehicle_Manoeuvre,Vehicle_Location-Restricted_Lane,Hit_Object_in_Carriageway,Hit_Object_off_Carriageway,Was_Vehicle_Left_Hand_Drive?,Age_of_Driver,Age_Band_of_Driver,Engine_Capacity_(CC),Propulsion_Code,Age_of_Vehicle,Location_Easting_OSGR,Location_Northing_OSGR,Longitude,Latitude,Police_Force,Number_of_Vehicles,Number_of_Casualties,Local_Authority_(District),1st_Road_Number,2nd_Road_Number,Urban_or_Rural_Area,Vehicle_Reference_df,Casualty_Reference,Sex_of_Casualty,Age_of_Casualty,Age_Band_of_Casualty,Pedestrian_Location,Pedestrian_Movement,Casualty_Type,Casualty_IMD_Decile,Sex_of_Driver
0,1.0,9.0,18.0,0.0,0.0,0.0,1.0,33.0,6.0,1895.0,1.0,16.0,408369.0,91104.0,-1.882813,50.719504,55.0,2.0,2.0,640.0,0.0,0.0,1.0,2.0,1.0,1.0,33.0,6.0,0.0,0.0,9.0,7.0,1.0
1,1.0,9.0,18.0,0.0,0.0,0.0,1.0,35.0,6.0,1560.0,2.0,5.0,423175.0,561511.0,-1.639723,54.947719,10.0,2.0,1.0,146.0,1.0,184.0,1.0,1.0,1.0,2.0,83.0,11.0,0.0,0.0,9.0,2.0,1.0
2,2.0,20.0,18.0,0.0,0.0,0.0,1.0,50.0,8.0,5880.0,2.0,7.0,522930.0,188580.0,-0.227170,51.582681,1.0,2.0,1.0,30.0,41.0,0.0,1.0,1.0,1.0,1.0,22.0,5.0,0.0,0.0,3.0,7.0,1.0
3,1.0,9.0,18.0,0.0,0.0,0.0,1.0,38.0,7.0,1896.0,2.0,13.0,441000.0,365280.0,-1.387939,53.183045,30.0,2.0,1.0,327.0,6038.0,0.0,1.0,2.0,1.0,1.0,29.0,6.0,0.0,0.0,9.0,8.0,1.0
4,1.0,9.0,18.0,0.0,0.0,0.0,1.0,22.0,5.0,1968.0,2.0,4.0,501310.0,229132.0,-0.527272,51.951463,40.0,2.0,1.0,424.0,5120.0,0.0,2.0,1.0,1.0,1.0,22.0,5.0,0.0,0.0,9.0,6.0,1.0


In [2]:
from typing import List, Any
import numpy as np

def clean_dataset(
    df: pd.DataFrame,
    cols_to_drop: List[Any] = None,
    missing_values: List[Any] = None
) -> pd.DataFrame:
    """
    Предобработка датафрейма с учетом пользовательских меток пропусков:
    1. Заполняет пропущенные значения
       - Модой для категориальных/текстовых признаков.
       - Средним значением для числовых признаков.
    2. Удаляет признаки, у которых одно значение занимает 100% строк.
    """
    df_cleaned = df.copy()
    n_rows = len(df_cleaned)
    
    if n_rows == 0:
        return df_cleaned

    # Задаем список меток для пропущенных значений по умолчанию, если не передан свой
    if missing_values is None:
        missing_values = ['', ' ', 'N/A', 'NA', 'null', 'nan', 'NaN', 'None', 'none', '?', '-']

    # 1. Приведение пользовательских пропусков к np.nan и их импутация
    for col in df_cleaned.columns:
        # Заменяем все кастомные метки пропусков на np.nan
        df_cleaned[col] = df_cleaned[col].replace(missing_values, np.nan)

        if df_cleaned[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_cleaned[col]):
                # а) Числовые переменные -> средним значением
                mean_val = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(mean_val)
            else:
                # б) Категориальные переменные -> модой
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])

    # 2. Проверка условий для удаления колонок
    for col in df_cleaned.columns:
        series = df_cleaned[col]
        max_value_count = series.value_counts().iloc[0]
        max_value_ratio = max_value_count / n_rows

        # Если одинаковых значений = 100% -> к удалению
        if max_value_ratio == 1:
            cols_to_drop.append(col)

    # Удаляем выявленные колонки
    if cols_to_drop:
        df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    return df_cleaned

df = clean_dataset(df)

In [3]:
df['Sex_of_Driver'] = df['Sex_of_Driver'].map({'2.0' : 1, '1.0' : 0})
y = df["Sex_of_Driver"]
X = df.drop(["Sex_of_Driver"], axis=1)

In [15]:
import time
import category_encoders as ce
from skrub import SimilarityEncoder, MinHashEncoder
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from k_fold_encoder import KFoldTargetEncoder
from k_fold_glmm_encoder import KFoldGLMMEncoder

cat_cols = ['Was_Vehicle_Left_Hand_Drive?','Urban_or_Rural_Area', 'Sex_of_Casualty']

encoders = {
    "MinHash": lambda: ColumnTransformer([(
                f"minhash_{col}",
                MinHashEncoder(),
                [col]
            )
            for col in cat_cols
        ],
        remainder="passthrough"
    ),
    "Similarity": lambda: ColumnTransformer([
        ("cat", SimilarityEncoder(), cat_cols)], 
        remainder="passthrough"),
   'Leave-one-out': lambda: ce.LeaveOneOutEncoder(cols=cat_cols),
   'K-fold':        lambda: KFoldTargetEncoder(cols=cat_cols),
   "CatBoost":      lambda: ce.CatBoostEncoder(cols=cat_cols),
   "CV-GLMM":       lambda: KFoldGLMMEncoder(cols=cat_cols),
   "WOE":           lambda: ce.WOEEncoder(cols=cat_cols),
   "JS":            lambda: ce.JamesSteinEncoder(cols=cat_cols),
   "MT":            lambda: ce.TargetEncoder(cols=cat_cols),
   'GLMM':          lambda: ce.GLMMEncoder(cols=cat_cols),
   'M-Estimate':    lambda: ce.MEstimateEncoder(cols=cat_cols),
   "OneHot":  lambda: ce.OneHotEncoder(cols=cat_cols),
   "Gray":    lambda: ce.GrayEncoder(cols=cat_cols),
   "BaseN":   lambda: ce.BaseNEncoder(cols=cat_cols),
   "Hashing": lambda: ce.HashingEncoder(cols=cat_cols,n_components=32),
   "Helmert": lambda: ce.HelmertEncoder(cols=cat_cols),
   "Sum":     lambda: ce.SumEncoder(cols=cat_cols),
   "BackwardDifference": lambda: ce.BackwardDifferenceEncoder(cols=cat_cols)
}

models = {
    # "LogReg": LogisticRegression(max_iter=1000),
    # "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    # "LinearSVM": LinearSVC(dual="auto", random_state=42),
    # "CART": DecisionTreeClassifier(random_state=42),
    # "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    # "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():
    for encoder_name, encoder in encoders.items():
        
        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            enc_instance = encoder()  # Вызываем lambda, получая объект кодировщика
            X_train_encoded = enc_instance.fit_transform(X_train, y_train)
            X_test_encoded = enc_instance.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Empirical",
                    "dataset": "RoadSafety",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Empirical, RoadSafety, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_road_safety_empirical.csv", index=True)
print("Done!")

C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, RoadSafety, LightGBM, MinHash, 1, 1, 0.8765617165619473, 0.8716779916272634, 0.7827584664250884, 0.7931152010906612, 0.7569941444372154, 0.8328561202576951, 1.7075262069702148


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, RoadSafety, LightGBM, MinHash, 1, 2, 0.8755154540055639, 0.8698197733156933, 0.7836979376370062, 0.7956034664975692, 0.7541272639846129, 0.8419074885926456, 1.712925910949707


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, RoadSafety, LightGBM, MinHash, 1, 3, 0.877258172775643, 0.8689973208435052, 0.7884753042233357, 0.7983107243409265, 0.7628403717593347, 0.8372405153901217, 1.6492960453033447


... (90 outputs left out)

C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, RoadSafety, LightGBM, Similarity, 5, 4, 0.8717578915013201, 0.8671399162080051, 0.7803775948460988, 0.7903480674781124, 0.7560258191028679, 0.8279348604151754, 1.3779423236846924


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, RoadSafety, LightGBM, Similarity, 5, 5, 0.8758612364848697, 0.8701715960818954, 0.7829277022190408, 0.7936723932641606, 0.7562398703403566, 0.8350035790980673, 1.3787841796875
Empirical, RoadSafety, LightGBM, Leave-one-out, 1, 1, 0.4449267325610952, 0.47513714239764615, 0.4995302643940411, 0.0030300329738882454, 0.37777777777777777, 0.0015211166785969936, 0.8509869575500488
Empirical, RoadSafety, LightGBM, Leave-one-out, 1, 2, 0.4551260132638934, 0.4835336430571665, 0.5003802621572049, 0.005520926090828139, 0.5849056603773585, 0.002773552831707972, 0.8552753925323486
Empirical, RoadSafety, LightGBM, Leave-one-out, 1, 3, 0.48852177869644997, 0.4903513544003997, 0.5005816034359342, 0.004281509232004281, 0.6857142857142857, 0.0021474588403722263, 0.8584022521972656
Empirical, RoadSafety, LightGBM, Leave-one-out, 1, 4, 0.4748169331049406, 0.4841089650780266, 0.5001342161775233, 0.00410018718245833, 0.5348837209302325, 0.00205798138869005, 0.8348720073699951
Empirical, RoadSafety